# 02 — The cascade, one step at a time

Notebook 01 called `Cascade.extract` and read the provenance off the end. This
one takes the same document and runs **each step by hand**, in order, on one
shared `Context`, so that the two things the cascade does become separable:

- what each step *produces*, and
- what the cascade *decides* about it.

Those are different, and keeping them apart is the single most expensive lesson
in this library's history. A step may have produced text **and** been refused,
and that text still competes for the document's text slot. Returning `None` for
a refusal — the obvious design — left **682 documents with zero characters while
the PDF had a text layer** (CLAUDE.md §5). `StepResult` therefore carries two
fields: the `attempt` (does the cascade stop?) and the `candidate` (does the text
enter the contest?).

By the end you will have seen a step refused, its candidate still alive, and the
contest at the bottom of the cascade choosing between candidates by usefulness
rather than by order of arrival.

In [ ]:
import logging

logging.disable(logging.INFO)  # the OCR backend's model-loading chatter

import pymupdf

from autosxtract import Config
from autosxtract.engines import available, get
from autosxtract.steps import Context, NativeStep, OCRStep, ScreeningStep, UnwrapStep

BODY = (
    "EXCELENTISSIMO SENHOR DOUTOR JUIZ DE DIREITO DA VARA CIVEL\n\n"
    "Peticao inicial nos autos do processo 0001234-56.2020.8.12.0001, em "
    "tramite perante a vara civel, em que o requerente pede a citacao do "
    "requerido na forma da decisao anterior, bem como a juntada dos documentos "
    "que seguem em anexo neste mesmo arquivo e cujo teor integra o pedido para "
    "todos os efeitos de direito. O exequente esclarece que a diligencia "
    "anterior restou infrutifera e que o oficial de justica certificou nos "
    "autos a impossibilidade de cumprimento."
)


def grey_block(width: int = 400, height: int = 300) -> bytes:
    """An actual raster image, because the coverage gate asks about images.

    A vector rectangle would look the same on screen and is *not* an image
    block — that is what made an early version of the test suite pass by
    accident.
    """
    pix = pymupdf.Pixmap(pymupdf.csGRAY, pymupdf.IRect(0, 0, width, height))
    pix.set_rect(pix.irect, (210,))
    for y in range(20, height - 20, 24):
        pix.set_rect(pymupdf.IRect(20, y, width - 20, y + 6), (40,))
    return pix.tobytes("png")


def filing_with_attachment() -> bytes:
    """Flawless text on the top half, a scanned attachment on the bottom."""
    doc = pymupdf.open()
    page = doc.new_page()
    page.insert_textbox(pymupdf.Rect(50, 50, 550, 380), BODY, fontsize=11)
    page.insert_image(pymupdf.Rect(60, 400, 540, 740), stream=grey_block())
    data = doc.tobytes()
    doc.close()
    return data


document = filing_with_attachment()
print(len(document), "bytes")

## The blackboard every step shares

`Context` exists so the whole cascade pays **once** for what is expensive and
shared: opening the PDF, reading the page profile, rasterising. A step that
rasterises on its own is not wrong, it is paying twice — and worse, two engines
would then be compared on different pixels, which turns the comparison into
preprocessing noise rather than evidence about the engines.

What a step is handed is the narrower `DocumentContext` view. Read the list of
what it *withholds*: `readings` and `texts` are the blackboard the consensus and
agreement gates decide on, and those gates belong to the cascade, not to a step.
A step reaching into them would be deciding on evidence it did not gather.

In [ ]:
ctx = Context(pdf_bytes=document, config=Config(), identifier="filing.pdf")

print("profile            :", ctx.profile)
print("visual content     :", ctx.profile.has_visual_content)
print("pages without text :", ctx.pages_without_text)

`pages_without_text` distinguishes three answers, and the routing depends on it:
`[]` means "no page is missing text, there is nothing to OCR"; a list means
"OCR only these"; and **`None` means "I could not read the structure"**, which
falls back to rasterising everything. Collapsing `None` into `[]` is the same
category error as treating a missing engine as an empty page.

## Stage 0 — `UnwrapStep`: is the file even a PDF?

It runs first and almost always does nothing: on a real PDF it costs the reading
of 16 bytes. It exists because in a real archive **128 documents arrived with a
`.pdf` extension and were not PDFs** — 16 plain RTF, 74 a proprietary signature
envelope, 38 PKCS#7 with the document inside. PyMuPDF raises on all of them and
no OCR helps: there is no image to recognise, there is plain text nobody was
reading.

In [ ]:
result = UnwrapStep().run(ctx)
print("accepted :", result.attempt.accepted)
print("reason   :", result.attempt.reason)
print("details  :", result.attempt.details)
print("candidate:", result.candidate)

"Refused" here means "carry on" — the file *is* a PDF, so stage 0 has nothing to
peel. Now the same step on a file whose extension lies. When the unwrap produces
**text**, the cascade stops there; when it produces **bytes** (a PDF that was
inside a signed envelope) the step swaps the context's content via
`replace_bytes` and the following steps measure the payload instead of the
wrapper.

In [ ]:
rtf_pretending_to_be_pdf = (
    rb"{\rtf1\ansi Peticao de folhas 12 nos autos do processo "
    rb"0001234-56.2020.8.12.0001, em que o requerente requer a citacao do "
    rb"requerido conforme decisao proferida pela vara civel.}"
)

envelope_ctx = Context(pdf_bytes=rtf_pretending_to_be_pdf, config=Config())
unwrapped = UnwrapStep().run(envelope_ctx)

print("accepted :", unwrapped.attempt.accepted)
print("reason   :", unwrapped.attempt.reason)
if unwrapped.candidate is not None:
    print("text     :", unwrapped.candidate.text[:90], "...")
else:
    # striprtf is an optional dependency; without it the step refuses in words
    # rather than raising, which is the library's rule for every missing tool.
    print("no candidate — the reason above says why")

## Step 1 — `NativeStep`: reading, not recognising

The text layer already inside the PDF costs ~13 ms per document (median 12.2 ms)
against ~400 ms for any OCR, and it resolves **31% of a real archive** on its
own. That asymmetry is the entire argument for a cascade: a single model for
everything charges OCR on the 31% of pages that already have the text ready.

Watch what happens on *this* document, though.

In [ ]:
native = NativeStep().run(ctx)

print("accepted  :", native.attempt.accepted)
print("reason    :", native.attempt.reason)
print("chars     :", native.attempt.chars)
print()
print("candidate :", native.candidate is not None)
print("score     :", native.candidate.score)
print("label     :", native.candidate.details["label"])

**This is the whole notebook in one cell.** The native text scored 0.95 — it is
excellent text — and the step was still refused, because the score describes the
text that *came out*, never the fraction of the page left behind. The coverage
gate saw a large image in a region with no text: in a filing that embeds an
official letter as an image, the native layer is flawless and the attachment —
the document's actual content — is never read.

And the candidate is still there. That is CLAUDE.md §5: the verdict decides
whether the cascade **stops**; the candidate enters the contest regardless. If
nothing downstream does better, this 0.95 text is what the document gets.

Note also what the step did on its way out: it recorded its reading on the
blackboard, refusal and all.

In [ ]:
print("texts    :", {k: v[:40] + "..." for k, v in ctx.texts.items()})
print("readings :", ctx.readings, " (useful words outside the stamp)")

Recording a *refused* reading is free at the point of decision and impossible
afterwards, and it is what turns "I could not read it" (one engine) into "there
is nothing here" (several). The consensus gate only means "this page is empty"
because every engine, including the ones that were turned down, left what it
read.

## Step 2 — `OCRStep`: one step, any engine

There is **no step per engine**. Vision and PP-OCRv6 go down exactly the same
code — rasterise, transcribe, measure, decide — and the only difference between
them is which object implements `transcribe_page`. That is what lets an engine be
added without touching the cascade, and it is the subject of notebook 05.

The step asks its engine for the contract (`interfaces.Engine`), not for a base
class. If this machine has no engine, the cell below substitutes a scripted
stand-in that implements the protocol and nothing else — announced, so nobody
mistakes its output for a real reading.

In [ ]:
from autosxtract.types import Transcription

READY = [info.name for info in available()]


class ScriptedEngine:
    """A stand-in for a machine with no OCR installed. Protocol only."""

    name = "scripted"
    scales_with_threads = True

    def available(self):
        return True, "scripted stand-in, not a real reading"

    def transcribe_page(self, image: bytes) -> tuple[str, float]:
        return BODY, 92.0

    def read_page(self, image: bytes):
        return None

    def recognize_crop(self, image: bytes):
        return None

    def transcribe(self, pages, *, parallelism=4, force_parallelism=False):
        return Transcription(
            text="\n\n".join(self.transcribe_page(p)[0] for p in pages),
            engine=self.name,
            pages_sent=len(pages),
            pages_answered=len(pages),
            mean_confidence=92.0,
        )

    def read_document(self, pdf_bytes, *, max_pages=3, min_reliable_words=3):
        return None


if READY:
    engine = get(READY[0])
    print("engine:", engine.name, "(real)")
else:
    engine = ScriptedEngine()
    print("engine:", engine.name, "— NO OCR ON THIS MACHINE, output below is invented")

ocr = OCRStep(engine).run(ctx)

print("accepted :", ocr.attempt.accepted)
print("reason   :", ocr.attempt.reason)
print("chars    :", ocr.attempt.chars, "| ms:", round(ocr.attempt.ms, 1))
for key, value in ocr.attempt.details.items():
    print(f"  {key:<12} {value}")

Three refusals live inside that step, in order of cost, and each has a distinct
provenance sentence:

1. **confidence below the floor** — a floor against degenerate output, never a
   quality criterion. Measured on 60 audited documents, engine confidence does
   not separate a good reading from an unsafe one; there was an unsafe document
   at confidence 100 (CLAUDE.md §7);
2. **incomplete reading** — `pages_answered < pages_sent` is a hole in the
   document, and a hole passes every volume test and disappears without a trace;
3. **the acceptance gate** — the single criterion, shared with the cascade, and
   the subject of notebook 03.

The `layers` entry in the details (present only when the engine exposes line
geometry) is the containment report: how much of the page is trustworthy and
where the holes are. A `skipped` value there is the library refusing to be silent
about a layer that did not run — without it, the absence of `[illegible]` markers
would look like a clean page.

## Step 3 — `ScreeningStep`: dropping on purpose, with evidence

This one reads no pixels at all. It looks at the text the earlier steps already
produced and asks whether the document is an identity or vehicle card, in which
case dropping it spares the expensive step **and** keeps tax numbers, parentage
and place of birth out of the output corpus.

The markers are not hand-picked: they come from 9,238 annotated background-text
regions of the BID dataset, filtered down to the ones appearing in **zero** of
486 legitimate documents. That filter is what dropped `REGISTRO GERAL` — four
false positives — and it is why the criterion that separates is density rather
than count: cards run 3.5 to 6.5 marks per thousand characters, and the
most-marked legitimate document sits at 1.56.

In [ ]:
card_ctx = Context(pdf_bytes=document, config=Config())
card_ctx.record_reading(
    "ocr",
    "REPUBLICA FEDERATIVA DO BRASIL CARTEIRA NACIONAL DE HABILITACAO "
    "DOC IDENTIDADE ORG EMISSOR FILIACAO VALIDADE PERMISSAO",
)

screened = ScreeningStep().run(card_ctx)
print("accepted :", screened.attempt.accepted)
print("reason   :", screened.attempt.reason)
print("evidence :", screened.attempt.details["evidence"])
print()
print(screened.candidate.text)

"Accepted" here means the cascade stops — with a notice in place of the text.
The original file is untouched; reprocessing is a matter of leaving the step out
of the cascade.

## The contest at the bottom

Now run the real thing over the same document and look at the last two lines of
the table. The cascade does not keep the text of the last step that ran: it keeps
the most **useful** candidate, where usefulness is `score × log(1 + volume)`.

Both dimensions, always. Volume alone lets a long unreadable OCR beat a short
correct reading; quality alone lets a 14-character placeholder — clean precisely
*because* it is short — beat the whole document. The logarithm damps volume on
purpose, so a candidate has to be *much* larger to make up for worse quality.

In [ ]:
from autosxtract import Cascade

cascade = Cascade(Config(), steps=[UnwrapStep(), NativeStep(), OCRStep(engine)])
final = cascade.extract(document, identifier="filing.pdf")

print(f"{'step':<22} {'ok':<6} {'chars':>7} {'ms':>8}  reason")
print("-" * 82)
for a in final.attempts:
    print(f"{a.step:<22} {str(a.accepted):<6} {a.chars:>7} {a.ms:>8.1f}  {a.reason}")

print()
print("winner   :", final.step, f"(score {final.score})")
print("discarded:", [(c.step, c.volume, round(c.usefulness, 2)) for c in final.discarded])

`discarded` is the losers of the contest, kept in the result rather than thrown
away — provenance again: whoever audits the extraction can see what the other
steps read and decide for themselves.

To make the point without depending on which engine this machine has, here is the
comparison in the abstract: a long, badly-read candidate against a short, clean
one.

In [ ]:
from autosxtract.quality.selection import pick
from autosxtract.types import Candidate

long_and_broken = Candidate(step="late_ocr", text="rn cl 1i " * 400, score=0.20)
short_and_clean = Candidate(step="native", text=BODY, score=0.95)

for c in (long_and_broken, short_and_clean):
    print(f"{c.step:<10} volume {c.volume:>5}  score {c.score:.2f}  usefulness {c.usefulness:.2f}")

print()
print("by volume alone :", max([long_and_broken, short_and_clean], key=lambda c: c.volume).step)
print("by usefulness   :", pick([long_and_broken, short_and_clean]).step)

That difference is the 12.7% of fall-through documents — 682 of them — that
ended with zero characters before the contest existed.

## What you now know

- a step returns a **verdict** and a **candidate**, and they are independent;
- a refused step still records its reading, which is what the cascade's gates
  later decide on;
- the coverage gate refuses text that is good *and* incomplete;
- the winner is the most useful candidate, not the last step to run.

Next: **03 — the two gates**, where "acceptable" gets its definition, and where
the stamp comes off before anything is measured.